# Comparacion de modelos para prediccion de series temporales

Este cuaderno replica la idea del taller original: predecir el siguiente valor de una serie temporal usando una ventana de valores anteriores.

Se comparan cinco arquitecturas:

- MLP
- RNN simple
- GRU
- LSTM
- Red densa simple

Mejoras incluidas:

- Semillas fijas para reproducibilidad.
- Normalizacion usando solo el tramo de entrenamiento para evitar fuga de informacion.
- Division temporal en entrenamiento, validacion y prueba.
- `shuffle=False` durante el entrenamiento, apropiado para series temporales.
- `EarlyStopping` y `ReduceLROnPlateau` para evitar entrenamiento innecesario.
- Tabla final con `MSE`, `RMSE` y `MAE` en la escala original de la serie.

In [1]:
# Librerias y configuracion inicial

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.layers import Input, Dense, SimpleRNN, GRU, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)

print("TensorFlow:", tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Generacion de la serie temporal sintetica

n = 600
t = np.arange(n)

tendencia = 0.005 * t
estacionalidad = 0.5 * np.cos(2 * np.pi * t / 50)
ruido = np.random.normal(loc=0, scale=0.1, size=n)

y = tendencia + estacionalidad + ruido

plt.plot(t, y, color="tab:blue", linewidth=1.6)
plt.title("Serie temporal original")
plt.xlabel("Tiempo")
plt.ylabel("Valor")
plt.show()

In [ ]:
# Division temporal y normalizacion sin fuga de informacion

window_size = 20
train_ratio = 0.80
train_end = int(n * train_ratio)

# Se ajusta la escala solamente con entrenamiento.
y_train_raw = y[:train_end]
y_min = y_train_raw.min()
y_max = y_train_raw.max()
y_range = y_max - y_min

y_scaled = (y - y_min) / y_range

print(f"Punto de corte entrenamiento/prueba: {train_end}")
print(f"Min entrenamiento: {y_min:.4f}")
print(f"Max entrenamiento: {y_max:.4f}")
print(f"Rango normalizado total: [{y_scaled.min():.4f}, {y_scaled.max():.4f}]")

plt.plot(t[:train_end], y_scaled[:train_end], label="Entrenamiento", color="tab:blue")
plt.plot(t[train_end:], y_scaled[train_end:], label="Prueba", color="tab:orange")
plt.axvline(train_end, color="black", linestyle="--", linewidth=1)
plt.title("Serie normalizada y division temporal")
plt.xlabel("Tiempo")
plt.ylabel("Valor normalizado")
plt.legend()
plt.show()

In [ ]:
# Creacion de ventanas supervisadas

# Cada fila de X contiene 20 valores consecutivos.
# Cada valor de Y es el siguiente punto despues de esa ventana.
def crear_ventanas(serie, window_size):
    X, Y, indices_objetivo = [], [], []
    for inicio in range(len(serie) - window_size):
        fin = inicio + window_size
        X.append(serie[inicio:fin])
        Y.append(serie[fin])
        indices_objetivo.append(fin)
    return np.array(X), np.array(Y), np.array(indices_objetivo)

X, Y, target_idx = crear_ventanas(y_scaled, window_size)

train_mask = target_idx < train_end
test_mask = target_idx >= train_end

x_train_full = X[train_mask]
y_train_full = Y[train_mask]
x_test = X[test_mask]
y_test = Y[test_mask]
test_time = target_idx[test_mask]

# Validacion temporal: se toma el ultimo 20% del entrenamiento.
val_ratio = 0.20
val_size = int(len(x_train_full) * val_ratio)

x_train = x_train_full[:-val_size]
y_train = y_train_full[:-val_size]
x_val = x_train_full[-val_size:]
y_val = y_train_full[-val_size:]

# Formato 3D requerido por RNN, GRU y LSTM: muestras, pasos temporales, variables.
x_train_seq = x_train.reshape(-1, window_size, 1)
x_val_seq = x_val.reshape(-1, window_size, 1)
x_test_seq = x_test.reshape(-1, window_size, 1)

print("X total:", X.shape)
print("Entrenamiento MLP/Densa:", x_train.shape)
print("Validacion MLP/Densa:", x_val.shape)
print("Prueba MLP/Densa:", x_test.shape)
print("Entrenamiento recurrente:", x_train_seq.shape)
print("Validacion recurrente:", x_val_seq.shape)
print("Prueba recurrente:", x_test_seq.shape)

## Definicion de modelos

Los modelos densos reciben una ventana plana de 20 valores. Los modelos recurrentes reciben la misma ventana con forma secuencial `(20, 1)`.

In [ ]:
# Constructores de modelos

def compilar(model):
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="mse"
    )
    return model


def crear_mlp(window_size):
    entrada = Input(shape=(window_size,), name="entrada_mlp")
    x = Dense(64, activation="relu")(entrada)
    x = Dense(32, activation="relu")(x)
    salida = Dense(1, activation="linear")(x)
    return compilar(Model(entrada, salida, name="MLP"))


def crear_rnn(window_size):
    entrada = Input(shape=(window_size, 1), name="entrada_rnn")
    x = SimpleRNN(32, activation="tanh")(entrada)
    salida = Dense(1, activation="linear")(x)
    return compilar(Model(entrada, salida, name="RNN_simple"))


def crear_gru(window_size):
    entrada = Input(shape=(window_size, 1), name="entrada_gru")
    x = GRU(32, activation="tanh")(entrada)
    salida = Dense(1, activation="linear")(x)
    return compilar(Model(entrada, salida, name="GRU"))


def crear_lstm(window_size):
    entrada = Input(shape=(window_size, 1), name="entrada_lstm")
    x = LSTM(32, activation="tanh")(entrada)
    salida = Dense(1, activation="linear")(x)
    return compilar(Model(entrada, salida, name="LSTM"))


def crear_densa_simple(window_size):
    entrada = Input(shape=(window_size,), name="entrada_densa_simple")
    x = Dense(32, activation="relu")(entrada)
    salida = Dense(1, activation="linear")(x)
    return compilar(Model(entrada, salida, name="Densa_simple"))

constructores = {
    "MLP": crear_mlp,
    "RNN simple": crear_rnn,
    "GRU": crear_gru,
    "LSTM": crear_lstm,
    "Densa simple": crear_densa_simple,
}

modelos_recurrentes = {"RNN simple", "GRU", "LSTM"}

In [ ]:
# Resumen de arquitecturas

for nombre, constructor in constructores.items():
    tf.keras.backend.clear_session()
    modelo = constructor(window_size)
    print("\n" + "=" * 70)
    print(nombre)
    modelo.summary()

## Entrenamiento y evaluacion

Se entrena cada modelo con los mismos datos, la misma ventana y la misma funcion de perdida. La evaluacion final se reporta en la escala original de la serie para que los errores sean interpretables.

In [ ]:
# Entrenamiento de todos los modelos

def desnormalizar(valores):
    return valores * y_range + y_min


def calcular_metricas(y_real, y_pred):
    error = y_real - y_pred
    mse = np.mean(error ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(error))
    return mse, rmse, mae

EPOCHS = 150
BATCH_SIZE = 32

historiales = {}
predicciones = {}
metricas = []

for nombre, constructor in constructores.items():
    print(f"\nEntrenando {nombre}...")
    tf.keras.backend.clear_session()
    modelo = constructor(window_size)

    if nombre in modelos_recurrentes:
        entrada_train, entrada_val, entrada_test = x_train_seq, x_val_seq, x_test_seq
    else:
        entrada_train, entrada_val, entrada_test = x_train, x_val, x_test

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True,
            verbose=0
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=7,
            min_lr=1e-5,
            verbose=0
        )
    ]

    history = modelo.fit(
        entrada_train,
        y_train,
        validation_data=(entrada_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=False,
        callbacks=callbacks,
        verbose=0
    )

    y_pred_scaled = modelo.predict(entrada_test, verbose=0).reshape(-1)
    y_pred = desnormalizar(y_pred_scaled)
    y_real = desnormalizar(y_test)

    mse, rmse, mae = calcular_metricas(y_real, y_pred)

    historiales[nombre] = history
    predicciones[nombre] = y_pred

    metricas.append({
        "Modelo": nombre,
        "Parametros": modelo.count_params(),
        "Epocas usadas": len(history.history["loss"]),
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "Mejor val_loss": np.min(history.history["val_loss"]),
    })

    print(f"{nombre}: RMSE={rmse:.4f}, MAE={mae:.4f}, epocas={len(history.history['loss'])}")

resultados = pd.DataFrame(metricas).sort_values("RMSE").reset_index(drop=True)
resultados

In [ ]:
# Grafica comparativa de predicciones en escala original

y_test_original = desnormalizar(y_test)

plt.plot(test_time, y_test_original, label="Real", color="black", linewidth=2.2)

for nombre, y_pred in predicciones.items():
    plt.plot(test_time, y_pred, label=nombre, linewidth=1.4, alpha=0.85)

plt.title("Comparacion de predicciones sobre el conjunto de prueba")
plt.xlabel("Tiempo")
plt.ylabel("Valor original")
plt.legend()
plt.show()

In [ ]:
# Curvas de perdida de entrenamiento y validacion

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.ravel()

for ax, (nombre, history) in zip(axes, historiales.items()):
    ax.plot(history.history["loss"], label="Entrenamiento")
    ax.plot(history.history["val_loss"], label="Validacion")
    ax.set_title(f"Perdida - {nombre}")
    ax.set_xlabel("Epoca")
    ax.set_ylabel("MSE")
    ax.legend()

for ax in axes[len(historiales):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Grafica del mejor modelo segun RMSE

mejor_modelo = resultados.iloc[0]["Modelo"]

plt.plot(test_time, y_test_original, label="Real", color="black", linewidth=2.2)
plt.plot(test_time, predicciones[mejor_modelo], label=f"Prediccion {mejor_modelo}", color="tab:red", linewidth=1.8)
plt.title(f"Mejor modelo segun RMSE: {mejor_modelo}")
plt.xlabel("Tiempo")
plt.ylabel("Valor original")
plt.legend()
plt.show()

print("Mejor modelo:", mejor_modelo)
display(resultados)

## Lectura de resultados

- `RMSE` penaliza mas los errores grandes y esta en la misma unidad de la serie original.
- `MAE` indica el error absoluto promedio.
- Si entrenamiento baja mucho pero validacion empeora, hay sobreajuste.
- En series temporales reales, conviene repetir el experimento con varias semillas, probar ventanas diferentes y validar con cortes temporales adicionales.